<a href="https://colab.research.google.com/github/kajal432/machine-learning-project/blob/main/Machine_learning_bigram_tensor_flow_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =============================================================================
# PART 1: SIMPLE BIGRAM MODEL
# A bigram model predicts the next word based on the PREVIOUS word only.
# Example: Given "I", what word comes next most often?
# =============================================================================

# -----------------------------------------------------------------------------
# Step 1: Import basic libraries
# -----------------------------------------------------------------------------
import random
import numpy
import pandas

from collections import defaultdict

# -----------------------------------------------------------------------------
# Step 2: Prepare our training data (sample sentences)
# -----------------------------------------------------------------------------
# These are the sentences our model will learn from.
# In real applications, you'd use thousands of sentences.

training_sentences = [
    "I love machine learning",
    "I love deep learning",
    "I love programming",
    "machine learning is amazing",
    "deep learning is powerful",
    "programming is fun",
    "I enjoy coding",
    "I enjoy learning new things",
    "learning is a journey",
]

# -----------------------------------------------------------------------------
# Step 3: Tokenization - Breaking sentences into words
# -----------------------------------------------------------------------------
# "Tokenization" means splitting text into individual pieces (tokens).
# Here, we split each sentence into a list of lowercase words.

def tokenize(sentence):
    """
    Convert a sentence into a list of lowercase words.

    Example:
        Input:  "I Love Pizza"
        Output: ["i", "love", "pizza"]
    """
    # .lower() makes everything lowercase (so "I" and "i" are treated the same)
    # .split() breaks the sentence at spaces
    return sentence.lower().split()

# Let's see tokenization in action:
print("=== Tokenization Example ===")
example = "I love machine learning"
print(f"Original: '{example}'")
print(f"Tokenized: {tokenize(example)}")
print()

# -----------------------------------------------------------------------------
# Step 4: Build the Bigram Model
# -----------------------------------------------------------------------------
# We'll count how often each word follows another word.
#
# Data structure: A dictionary of dictionaries
#   bigram_counts["i"]["love"] = 3  means "love" followed "i" three times
#
# defaultdict automatically creates empty entries when we access new keys.

def build_bigram_model(sentences):
    """
    Build a bigram model from a list of sentences.

    Returns a dictionary where:
        - Keys are words
        - Values are dictionaries of {next_word: count}

    Example output structure:
        {
            "i": {"love": 3, "enjoy": 2},
            "love": {"machine": 1, "deep": 1, "programming": 1}
        }
    """
    # defaultdict(lambda: defaultdict(int)) creates a nested dictionary
    # that automatically initializes missing keys with 0
    bigram_counts = defaultdict(lambda: defaultdict(int))

    for sentence in sentences:
        # Tokenize the sentence
        words = tokenize(sentence)

        # Go through each pair of consecutive words
        # range(len(words) - 1) stops one before the end
        # because we need pairs: (word[i], word[i+1])
        for i in range(len(words) - 1):
            current_word = words[i]      # The word we're looking at
            next_word = words[i + 1]     # The word that follows it

            # Increment the count for this pair
            bigram_counts[current_word][next_word] += 1

    return bigram_counts

# Build our model
print("=== Building Bigram Model ===")
bigram_model = build_bigram_model(training_sentences)

# Let's see what words follow "i":
print("Words that follow 'i':")
for next_word, count in bigram_model["i"].items():
    print(f"  'i' → '{next_word}': {count} times")
print()

# -----------------------------------------------------------------------------
# Step 5: Convert Counts to Probabilities
# -----------------------------------------------------------------------------
# Instead of raw counts, we want probabilities.
# Probability = count of specific pair / total count of all pairs starting with that word
#
# Example: If "i" is followed by "love" 3 times and "enjoy" 2 times,
#          P("love" | "i") = 3/5 = 0.6
#          P("enjoy" | "i") = 2/5 = 0.4

def counts_to_probabilities(bigram_counts):
    """
    Convert raw counts to probabilities.

    Returns a dictionary where:
        - Keys are words
        - Values are dictionaries of {next_word: probability}
    """
    bigram_probs = {}

    for word, next_words in bigram_counts.items():
        # Calculate total count for all words following this word
        total_count = sum(next_words.values())

        # Convert each count to a probability
        bigram_probs[word] = {
            next_word: count / total_count
            for next_word, count in next_words.items()
        }

    return bigram_probs

bigram_probs = counts_to_probabilities(bigram_model)

print("=== Bigram Probabilities ===")
print("Probabilities for words following 'i':")
for next_word, prob in bigram_probs["i"].items():
    print(f"  P('{next_word}' | 'i') = {prob:.2f}")
print()

# -----------------------------------------------------------------------------
# Step 6: Predict the Next Word
# -----------------------------------------------------------------------------
# Given a word, predict the most likely next word.

def predict_next_word_bigram(word, bigram_probs):
    """
    Predict the most likely next word given the current word.

    Args:
        word: The current word (string)
        bigram_probs: The probability model (dictionary)

    Returns:
        The most probable next word, or None if word is unknown
    """
    word = word.lower()

    # Check if we've seen this word before
    if word not in bigram_probs:
        return None  # Unknown word

    # Find the next word with highest probability
    next_words = bigram_probs[word]
    best_word = max(next_words, key=next_words.get)
    best_prob = next_words[best_word]

    return best_word, best_prob

# Test predictions
print("=== Bigram Predictions ===")
test_words = ["i", "love", "learning", "programming"]
for word in test_words:
    result = predict_next_word_bigram(word, bigram_probs)
    if result:
        next_word, prob = result
        print(f"After '{word}' → '{next_word}' (probability: {prob:.2f})")
    else:
        print(f"After '{word}' → Unknown word!")
print()

# -----------------------------------------------------------------------------
# Step 7: Generate a Sentence
# -----------------------------------------------------------------------------
# Start with a word and keep predicting the next word.

def generate_sentence_bigram(start_word, bigram_probs, max_length=10):
    """
    Generate a sentence starting with the given word.

    Uses random selection weighted by probabilities for variety.
    """
    sentence = [start_word.lower()]
    current_word = start_word.lower()

    for _ in range(max_length - 1):  # -1 because we already have the start word
        if current_word not in bigram_probs:
            break  # Can't continue, unknown word

        # Get possible next words and their probabilities
        next_words = list(bigram_probs[current_word].keys())
        probabilities = list(bigram_probs[current_word].values())

        # Randomly choose next word (weighted by probability)
        # This adds variety - otherwise we'd always get the same sentence
        current_word = random.choices(next_words, weights=probabilities)[0]
        sentence.append(current_word)

    return " ".join(sentence)

print("=== Generated Sentences (Bigram) ===")
for i in range(3):
    generated = generate_sentence_bigram("i", bigram_probs, max_length=5)
    print(f"  {i+1}. {generated}")
print()


=== Tokenization Example ===
Original: 'I love machine learning'
Tokenized: ['i', 'love', 'machine', 'learning']

=== Building Bigram Model ===
Words that follow 'i':
  'i' → 'love': 3 times
  'i' → 'enjoy': 2 times

=== Bigram Probabilities ===
Probabilities for words following 'i':
  P('love' | 'i') = 0.60
  P('enjoy' | 'i') = 0.40

=== Bigram Predictions ===
After 'i' → 'love' (probability: 0.60)
After 'love' → 'machine' (probability: 0.33)
After 'learning' → 'is' (probability: 0.75)
After 'programming' → 'is' (probability: 1.00)

=== Generated Sentences (Bigram) ===
  1. i love programming is powerful
  2. i enjoy coding
  3. i enjoy learning is amazing



In [ ]:
# =============================================================================
# PART 1: SIMPLE BIGRAM MODEL
# A bigram model predicts the next word based on the PREVIOUS word only.
# Example: Given "I", what word comes next most often?
# =============================================================================

# -----------------------------------------------------------------------------
# Step 1: Import basic libraries
# -----------------------------------------------------------------------------
import random
import tensorflow
import numpy
import pandas
from collections import defaultdict

# -----------------------------------------------------------------------------
# Step 2: Prepare our training data (sample sentences)
# -----------------------------------------------------------------------------
# These are the sentences our model will learn from.
# In real applications, you'd use thousands of sentences.

training_sentences = [
    "I love machine learning",
    "I love deep learning",
    "I love programming",
    "machine learning is amazing",
    "deep learning is powerful",
    "programming is fun",
    "I enjoy coding",
    "I enjoy learning new things",
    "learning is a journey",
]

# -----------------------------------------------------------------------------
# Step 3: Tokenization - Breaking sentences into words
# -----------------------------------------------------------------------------
# "Tokenization" means splitting text into individual pieces (tokens).
# Here, we split each sentence into a list of lowercase words.

def tokenize(sentence):
    """
    Convert a sentence into a list of lowercase words.

    Example:
        Input:  "I Love Pizza"
        Output: ["i", "love", "pizza"]
    """
    # .lower() makes everything lowercase (so "I" and "i" are treated the same)
    # .split() breaks the sentence at spaces
    return sentence.lower().split()

# Let's see tokenization in action:
print("=== Tokenization Example ===")
example = "I love machine learning"
print(f"Original: '{example}'")
print(f"Tokenized: {tokenize(example)}")
print()

# -----------------------------------------------------------------------------
# Step 4: Build the Bigram Model
# -----------------------------------------------------------------------------
# We'll count how often each word follows another word.
#
# Data structure: A dictionary of dictionaries
#   bigram_counts["i"]["love"] = 3  means "love" followed "i" three times
#
# defaultdict automatically creates empty entries when we access new keys.

def build_bigram_model(sentences):
    """
    Build a bigram model from a list of sentences.

    Returns a dictionary where:
        - Keys are words
        - Values are dictionaries of {next_word: count}

    Example output structure:
        {
            "i": {"love": 3, "enjoy": 2},
            "love": {"machine": 1, "deep": 1, "programming": 1}
        }
    """
    # defaultdict(lambda: defaultdict(int)) creates a nested dictionary
    # that automatically initializes missing keys with 0
    bigram_counts = defaultdict(lambda: defaultdict(int))

    for sentence in sentences:
        # Tokenize the sentence
        words = tokenize(sentence)

        # Go through each pair of consecutive words
        # range(len(words) - 1) stops one before the end
        # because we need pairs: (word[i], word[i+1])
        for i in range(len(words) - 1):
            current_word = words[i]      # The word we're looking at
            next_word = words[i + 1]     # The word that follows it

            # Increment the count for this pair
            bigram_counts[current_word][next_word] += 1

    return bigram_counts

# Build our model
print("=== Building Bigram Model ===")
bigram_model = build_bigram_model(training_sentences)

# Let's see what words follow "i":
print("Words that follow 'i':")
for next_word, count in bigram_model["i"].items():
    print(f"  'i' → '{next_word}': {count} times")
print()

# -----------------------------------------------------------------------------
# Step 5: Convert Counts to Probabilities
# -----------------------------------------------------------------------------
# Instead of raw counts, we want probabilities.
# Probability = count of specific pair / total count of all pairs starting with that word
#
# Example: If "i" is followed by "love" 3 times and "enjoy" 2 times,
#          P("love" | "i") = 3/5 = 0.6
#          P("enjoy" | "i") = 2/5 = 0.4

def counts_to_probabilities(bigram_counts):
    """
    Convert raw counts to probabilities.

    Returns a dictionary where:
        - Keys are words
        - Values are dictionaries of {next_word: probability}
    """
    bigram_probs = {}

    for word, next_words in bigram_counts.items():
        # Calculate total count for all words following this word
        total_count = sum(next_words.values())

        # Convert each count to a probability
        bigram_probs[word] = {
            next_word: count / total_count
            for next_word, count in next_words.items()
        }

    return bigram_probs

bigram_probs = counts_to_probabilities(bigram_model)

print("=== Bigram Probabilities ===")
print("Probabilities for words following 'i':")
for next_word, prob in bigram_probs["i"].items():
    print(f"  P('{next_word}' | 'i') = {prob:.2f}")
print()

# -----------------------------------------------------------------------------
# Step 6: Predict the Next Word
# -----------------------------------------------------------------------------
# Given a word, predict the most likely next word.

def predict_next_word_bigram(word, bigram_probs):
    """
    Predict the most likely next word given the current word.

    Args:
        word: The current word (string)
        bigram_probs: The probability model (dictionary)

    Returns:
        The most probable next word, or None if word is unknown
    """
    word = word.lower()

    # Check if we've seen this word before
    if word not in bigram_probs:
        return None  # Unknown word

    # Find the next word with highest probability
    next_words = bigram_probs[word]
    best_word = max(next_words, key=next_words.get)
    best_prob = next_words[best_word]

    return best_word, best_prob

# Test predictions
print("=== Bigram Predictions ===")
test_words = ["i", "love", "learning", "programming"]
for word in test_words:
    result = predict_next_word_bigram(word, bigram_probs)
    if result:
        next_word, prob = result
        print(f"After '{word}' → '{next_word}' (probability: {prob:.2f})")
    else:
        print(f"After '{word}' → Unknown word!")
print()

# -----------------------------------------------------------------------------
# Step 7: Generate a Sentence
# -----------------------------------------------------------------------------
# Start with a word and keep predicting the next word.

def generate_sentence_bigram(start_word, bigram_probs, max_length=10):
    """
    Generate a sentence starting with the given word.

    Uses random selection weighted by probabilities for variety.
    """
    sentence = [start_word.lower()]
    current_word = start_word.lower()

    for _ in range(max_length - 1):  # -1 because we already have the start word
        if current_word not in bigram_probs:
            break  # Can't continue, unknown word

        # Get possible next words and their probabilities
        next_words = list(bigram_probs[current_word].keys())
        probabilities = list(bigram_probs[current_word].values())

        # Randomly choose next word (weighted by probability)
        # This adds variety - otherwise we'd always get the same sentence
        current_word = random.choices(next_words, weights=probabilities)[0]
        sentence.append(current_word)

    return " ".join(sentence)

print("=== Generated Sentences (Bigram) ===")
for i in range(3):
    generated = generate_sentence_bigram("i", bigram_probs, max_length=5)
    print(f"  {i+1}. {generated}")
print()


In [ ]:
# =============================================================================
# PART 3: DEEP LEARNING MODEL WITH TENSORFLOW
# A neural network that learns word patterns and predicts the next word.
# =============================================================================

print("\n" + "="*60)
print("DEEP LEARNING MODEL (TensorFlow)")
print("="*60 + "\n")

# -----------------------------------------------------------------------------
# Step 1: Import TensorFlow and Related Libraries
# -----------------------------------------------------------------------------
import numpy as np  # For numerical operations

# TensorFlow is a deep learning library created by Google
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# Set random seed for reproducibility (same results each run)
tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow version: {tf.__version__}")
print()

# -----------------------------------------------------------------------------
# Step 2: Prepare More Training Data
# -----------------------------------------------------------------------------
# Neural networks need more data to learn patterns.
# In real applications, you'd use thousands or millions of sentences.

training_text = """
I love machine learning and deep learning
Machine learning is a subset of artificial intelligence
Deep learning uses neural networks with many layers
I enjoy programming in Python
Python is great for machine learning
Neural networks can learn complex patterns
I love to code and build projects
Artificial intelligence is transforming the world
Learning to code is a valuable skill
I enjoy building machine learning models
Deep learning models need lots of data
Programming is both fun and challenging
I love creating new applications
Machine learning models can make predictions
The future of AI is exciting
"""

# Split into sentences
sentences = [s.strip() for s in training_text.strip().split('\n') if s.strip()]
print(f"Number of training sentences: {len(sentences)}")
print(f"First 3 sentences: {sentences[:3]}")
print()

# -----------------------------------------------------------------------------
# Step 3: Tokenization with Keras Tokenizer
# -----------------------------------------------------------------------------
# The Tokenizer converts words to numbers and vice versa.
# Neural networks work with numbers, not text!

# Create a tokenizer
tokenizer = Tokenizer()

# Fit the tokenizer on our sentences
# This builds a vocabulary: each unique word gets a unique number
tokenizer.fit_on_texts(sentences)

# Get the vocabulary size (total number of unique words + 1 for padding)
vocab_size = len(tokenizer.word_index) + 1

print("=== Vocabulary ===")
print(f"Total unique words: {vocab_size - 1}")
print(f"Word to number mapping (first 10):")
for word, idx in list(tokenizer.word_index.items())[:10]:
    print(f"  '{word}' → {idx}")
print()

# -----------------------------------------------------------------------------
# Step 4: Create Training Sequences
# -----------------------------------------------------------------------------
# We'll create input-output pairs:
#   Input: sequence of words → Output: the next word
#
# Example: "I love machine learning"
#   Pairs: [I] → love
#          [I, love] → machine
#          [I, love, machine] → learning

input_sequences = []

for sentence in sentences:
    # Convert sentence to sequence of numbers
    token_list = tokenizer.texts_to_sequences([sentence])[0]

    # Create n-gram sequences
    # [1, 2, 3, 4] → [[1,2], [1,2,3], [1,2,3,4]]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

print("=== Sample Input Sequences ===")
print("First 5 sequences (as numbers):")
for seq in input_sequences[:5]:
    # Convert numbers back to words for display
    words = [tokenizer.index_word.get(idx, '?') for idx in seq]
    print(f"  {seq} → {words}")
print()

# -----------------------------------------------------------------------------
# Step 5: Pad Sequences (Make Them Same Length)
# -----------------------------------------------------------------------------
# Neural networks need fixed-size inputs.
# We pad shorter sequences with zeros at the beginning.
#
# Example: [[1,2], [1,2,3], [1,2,3,4]]
# Padded:  [[0,0,1,2], [0,1,2,3], [1,2,3,4]]

# Find the maximum sequence length
max_sequence_length = max(len(seq) for seq in input_sequences)
print(f"Maximum sequence length: {max_sequence_length}")

# Pad all sequences to the same length
# pre-padding: add zeros at the beginning (default)
input_sequences = pad_sequences(input_sequences,
                                 maxlen=max_sequence_length,
                                 padding='pre')

print(f"Shape of padded sequences: {input_sequences.shape}")
print("First 3 padded sequences:")
for seq in input_sequences[:3]:
    print(f"  {seq}")
print()

# -----------------------------------------------------------------------------
# Step 6: Split into Features (X) and Labels (y)
# -----------------------------------------------------------------------------
# X (input): All words except the last one
# y (output): The last word (what we want to predict)

# Convert to numpy array for manipulation
input_sequences = np.array(input_sequences)

# X = all columns except the last
X = input_sequences[:, :-1]

# y = only the last column
y = input_sequences[:, -1]

print("=== Features (X) and Labels (y) ===")
print(f"X shape: {X.shape} (samples, sequence_length)")
print(f"y shape: {y.shape} (samples,)")
print()
print("Example - First 3 samples:")
for i in range(3):
    x_words = [tokenizer.index_word.get(idx, '_') for idx in X[i] if idx != 0]
    y_word = tokenizer.index_word.get(y[i], '?')
    print(f"  Input: {x_words} → Predict: '{y_word}'")
print()

# -----------------------------------------------------------------------------
# Step 7: One-Hot Encode the Labels
# -----------------------------------------------------------------------------
# Neural networks output probabilities for each possible word.
# We convert labels to "one-hot" format.
#
# Example: If y=3 and vocab_size=5
# One-hot: [0, 0, 0, 1, 0]  (1 at position 3, 0 elsewhere)

y_one_hot = tf.keras.utils.to_categorical(y, num_classes=vocab_size)
print(f"y_one_hot shape: {y_one_hot.shape} (samples, vocab_size)")
print()

# -----------------------------------------------------------------------------
# Step 8: Build the Neural Network Model
# -----------------------------------------------------------------------------
# We'll use an LSTM (Long Short-Term Memory) network.
# LSTM is great for sequences because it can remember earlier words.

print("=== Building Neural Network ===")

model = Sequential([
    # Layer 1: Embedding Layer
    # Converts word indices to dense vectors
    # Input: word index (like 5)
    # Output: vector of 50 numbers representing that word
    # Think of it as: each word becomes a point in 50-dimensional space
    # Similar words will be near each other
    Embedding(input_dim=vocab_size,      # Size of vocabulary
              output_dim=50,              # Size of word vectors
              input_length=max_sequence_length - 1),  # Length of input sequences

    # Layer 2: LSTM Layer
    # Processes the sequence and learns patterns
    # 100 units = 100 "memory cells"
    # return_sequences=False means we only want the final output
    LSTM(units=100),

    # Layer 3: Dropout Layer
    # Randomly "turns off" 20% of neurons during training
    # This prevents overfitting (memorizing instead of learning)
    Dropout(0.2),

    # Layer 4: Dense (Output) Layer
    # Outputs probability for each word in vocabulary
    # softmax: converts raw scores to probabilities (sum to 1)
    Dense(units=vocab_size, activation='softmax')
])

# Compile the model
# optimizer: how the model updates its weights (Adam is popular)
# loss: what the model tries to minimize (categorical_crossentropy for classification)
# metrics: what we track during training (accuracy)
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Print model summary
print("\nModel Architecture:")
model.summary()
print()

# -----------------------------------------------------------------------------
# Step 9: Train the Model
# -----------------------------------------------------------------------------
# The model will see the data multiple times (epochs) and improve.

print("=== Training the Model ===")
print("This may take a moment...\n")

history = model.fit(
    X,              # Input sequences
    y_one_hot,      # Target words (one-hot encoded)
    epochs=100,     # Number of times to go through all data
    batch_size=32,  # Number of samples to process at once
    verbose=1       # Show progress (1=progress bar, 0=silent)
)

print("\nTraining complete!")
print(f"Final accuracy: {history.history['accuracy'][-1]:.2%}")
print()

# -----------------------------------------------------------------------------
# Step 10: Make Predictions
# -----------------------------------------------------------------------------

def predict_next_word_nn(text, model, tokenizer, max_length):
    """
    Predict the next word using the neural network.

    Args:
        text: The input text (string)
        model: Trained Keras model
        tokenizer: Fitted Keras tokenizer
        max_length: Maximum sequence length

    Returns:
        Predicted word and probability
    """
    # Convert text to sequence of numbers
    sequence = tokenizer.texts_to_sequences([text])[0]

    # Pad to the correct length
    sequence = pad_sequences([sequence], maxlen=max_length - 1, padding='pre')

    # Get prediction probabilities for each word
    predictions = model.predict(sequence, verbose=0)[0]

    # Find the word with highest probability
    predicted_idx = np.argmax(predictions)
    predicted_word = tokenizer.index_word.get(predicted_idx, '<unknown>')
    confidence = predictions[predicted_idx]

    return predicted_word, confidence

print("=== Neural Network Predictions ===")
test_phrases = [
    "i love",
    "machine learning",
    "deep learning",
    "i enjoy",
    "python is",
]

for phrase in test_phrases:
    word, conf = predict_next_word_nn(phrase, model, tokenizer, max_sequence_length)
    print(f"'{phrase}' → '{word}' (confidence: {conf:.2%})")
print()

# -----------------------------------------------------------------------------
# Step 11: Generate Text with the Neural Network
# -----------------------------------------------------------------------------

def generate_text_nn(seed_text, model, tokenizer, max_length, num_words=5):
    """
    Generate text by repeatedly predicting the next word.

    Args:
        seed_text: Starting text
        model: Trained model
        tokenizer: Fitted tokenizer
        max_length: Maximum sequence length
        num_words: How many words to generate

    Returns:
        Generated text
    """
    result = seed_text

    for _ in range(num_words):
        # Predict next word
        next_word, _ = predict_next_word_nn(result, model, tokenizer, max_length)

        # Append to result
        result = result + " " + next_word

    return result

print("=== Generated Text (Neural Network) ===")
seed_phrases = ["i love", "machine learning", "the future"]

for seed in seed_phrases:
    generated = generate_text_nn(seed, model, tokenizer, max_sequence_length, num_words=5)
    print(f"Starting with '{seed}':")
    print(f"  → {generated}")
print()

# -----------------------------------------------------------------------------
# Step 12: Get Top-K Predictions
# -----------------------------------------------------------------------------
# Sometimes we want to see the top few predictions, not just the best one.

def get_top_k_predictions(text, model, tokenizer, max_length, k=5):
    """
    Get the top K most likely next words.
    """
    sequence = tokenizer.texts_to_sequences([text])[0]
    sequence = pad_sequences([sequence], maxlen=max_length - 1, padding='pre')

    predictions = model.predict(sequence, verbose=0)[0]

    # Get indices of top k predictions
    top_k_indices = np.argsort(predictions)[-k:][::-1]

    results = []
    for idx in top_k_indices:
        word = tokenizer.index_word.get(idx, '<unknown>')
        prob = predictions[idx]
        results.append((word, prob))

    return results

print("=== Top-5 Predictions ===")
test_phrase = "i love"
print(f"Top 5 predictions after '{test_phrase}':")
top_predictions = get_top_k_predictions(test_phrase, model, tokenizer, max_sequence_length, k=5)
for word, prob in top_predictions:
    print(f"  '{word}': {prob:.2%}")
print()

# =============================================================================
# SUMMARY
# =============================================================================
print("="*60)
print("SUMMARY: What We Built")
print("="*60)
print("""
1. BIGRAM MODEL (Simple)
   - Predicts next word based on PREVIOUS word only
   - Uses word-pair frequencies
   - Simple but limited context

2. TRIGRAM MODEL (Simple)
   - Predicts next word based on PREVIOUS TWO words
   - Better context than bigram
   - Still limited by fixed window

3. DEEP LEARNING MODEL (TensorFlow/Keras)
   - Uses LSTM neural network
   - Can learn complex patterns
   - Better at understanding context
   - Components:
     * Embedding Layer: Words → Vectors
     * LSTM Layer: Learns sequence patterns
     * Dropout Layer: Prevents overfitting
     * Dense Layer: Outputs word probabilities

Key Differences:
- N-gram models: Fast, interpretable, limited context
- Deep Learning: Slower to train, but learns rich patterns
""")
